## Orchestrating Multi Skill Workflows

# Unit 16: Multi-Skill Architecture and Pipeline Interoperability

## Introduction

Welcome to Unit 4 of Skills — Extending Claude's Capabilities! Throughout this course, we have explored the foundational mechanics of the Skills system, engineered practical capability modules from scratch, and mastered multi-tool design patterns. Now we are ready to tackle the final piece of the architectural puzzle: designing independent Skills that interoperate seamlessly within a unified processing pipeline.

In this lesson, we will step through a realistic enterprise data-mining scenario: ingestion and trend isolation within a raw `sales_data.csv` file, automated generation of publication-quality data graphics, and compiling a structured PDF executive report. Along the way, you will discover how to establish strict input/output contract boundaries between distinct modules, trace and eliminate activation collisions when multiple capabilities compete for a single user prompt, and determine when a workflow requires advanced task isolation hooks.

Let's unlock the true power of multi-skill workflows!

---

## The Core Concept of Multi-Skill Pipelines

In previous modules, we observed individual custom Skills solving isolated, single-purpose development problems—such as mapping localized layout constraints, testing localized API endpoints, or extracting specific database arrays. Real-world business operations, however, are rarely linear. They require discrete, specialized capabilities to hand off processing states sequentially:

```text
 ┌─────────────────────────┐      ┌─────────────────────────┐      ┌─────────────────────────┐
 │    1. Data Isolation    │ ───► │    2. Visualization     │ ───► │   3. Document Assembly  │
 │ (Statistical Extraction)│      │  (Vector Rendering)     │      │ (ReportLab Compilation) │
 └─────────────────────────┘      └─────────────────────────┘      └─────────────────────────┘

```

When building automated systems that span multiple operational domains (such as data analysis, programmatic visualization, and document compilation), you do not bundle all execution steps into a single, massive `SKILL.md` file. That anti-pattern induces context fatigue and makes maintenance highly brittle. Instead, you deploy an ecosystem of modular, highly focused Skills designed to complement each other by producing file outputs that perfectly fulfill the input specifications of downstream modules.

---

## Execution Trace of a Coordinated Pipeline

Let's look at how Claude orchestrates multiple specialized capabilities simultaneously when faced with a multi-layered production request:

```text
❯ I need to analyze sales_data.csv, create publication-quality charts, and generate a PDF report with findings

```

### Phase 1: Ingestion and Trend Analysis

Claude begins the loop by initializing its read brokers to parse the data structure. If a localized data-mining module isn't available, the agent uses standard repository reading tools to extract core parameters:

```text
● Read(sales_data.csv)  ⎿ Read 1000 lines from sales_data.csv

● Key findings isolated:
  - Revenue trend: +15% YoY
  - Top performing region: North (40% of sales volume)
  - Peak operational month: December
  - Mean order value: $127.50

```

### Phase 2: Intercepting the Graphical Visualization Step

As the text generation loop hits the charting requirement, the natural language prompt router parses the phrase *"create publication-quality charts"*. This safely fires your pre-compiled `sci-viz` metadata hooks, mounting the visualization layout engine:

```text
● [Skill selected: sci-viz]
● Creating publication-quality visualization canvas strings...

```

The mounted capability immediately locks down the script parameters, enforcing a high-resolution canvas profile, light gray layout grids, and colorblind-safe color matrices before outputting dual-format preview assets:

```python
import matplotlib.pyplot as plt
import pandas as pd

# Enforcing strict publication design variables
plt.rcParams['figure.dpi'] = 300
plt.rcParams['font.size'] = 10

df = pd.read_csv('sales_data.csv')
fig, ax = plt.subplots(figsize=(6, 4))

monthly = df.groupby('month')['revenue'].sum()
ax.plot(monthly.index, monthly.values, marker='o')
ax.set_xlabel('Month')
ax.set_ylabel('Revenue (USD)')
ax.set_title('Monthly Revenue Trend')
ax.grid(True, alpha=0.3, color='gray')

plt.tight_layout()
plt.savefig('revenue_trend.png', dpi=300, bbox_inches='tight')
plt.savefig('revenue_trend.pdf', bbox_inches='tight')

```

### Phase 3: Programmatic PDF Document Assembly

With the high-resolution charts written to the local workspace, the final stage uses the system's document generation tools to assemble a polished, executive-ready PDF asset:

```python
from reportlab.lib.pagesizes import letter
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, Image
from reportlab.lib.styles import getSampleStyleSheet
from reportlab.lib.units import inch

doc = SimpleDocTemplate("sales_report.pdf", pagesize=letter)
story = []
styles = getSampleStyleSheet()

# Structural Layout Construction
story.append(Paragraph("Sales Analysis Report", styles['Title']))
story.append(Spacer(1, 0.3 * inch))
story.append(Paragraph("<b>Executive Summary</b><br/>...", styles['Normal']))

# Programmatically embedding the chart generated by the previous phase
story.append(Paragraph("<b>Revenue Trend</b>", styles['Heading2']))
story.append(Image('revenue_trend.png', width=5 * inch, height=3.33 * inch))

story.append(Paragraph("<b>Recommendations</b><br/>...", styles['Normal']))
doc.build(story)

```

The multi-stage execution closes cleanly, leaving structured, production-ready binary artifacts on your disk without manual step-by-step guidance:

```text
● Complete workflow finished!  
  Files created:
  - revenue_trend.png/pdf (300 DPI, publication-ready vector assets)
  - sales_report.pdf (3 pages, professional typographic alignment)

```

---

## Designing for Interoperable Input/Output Contracts

To link separate capabilities into an automated multi-skill pipeline, you must explicitly align their input and output parameters during design time.

Let's look at the layout requirements for an independent document-assembly module designed to interface seamlessly with our existing graphics generator:

### Target File Path

```text
.claude/skills/pdf-report/SKILL.md

```

### Complete Code Blueprint: `SKILL.md`

```markdown
---
name: pdf-report
description: Generate professional PDF reports with charts, tables, and formatted text
allowed-tools: Write, Read, Bash
---

# PDF Report Generator Skill

## When to Use
- User asks for an automated PDF report or summary document
- User wants to package local repository findings or analytics data
- User needs formatted document structures embedding external charts

## Standards
1. Compile documents using the ReportLab library layout engine
2. Enforce a mandatory structural flow: Title -> Executive Summary -> Mounted Charts -> Recommendations
3. Maintain high-resolution formatting targets:
   - Body typography: 12pt; Section headers: 16pt bold
   - Embedded image assets must map directly to 300 DPI source graphs

```

### Engineering Compatibility Links:

Notice the intentional cross-module layout link here:

* **`sci-viz` Module Output:** Explicitly builds and saves high-density image files hardcoded to a minimum resolution of **300 DPI**.
* **`pdf-report` Module Input:** Explicitly maps document images assuming a base target density of **300 DPI**.

This compatibility isn't an accident—it's the result of clean technical planning. By aligning artifact expectations across distinct modules, you allow Claude Code to smoothly transition data through multi-stage pipelines without file corruption or scaling anomalies.

---

## Mitigating Selection Collisions

When an active project repository houses dozens of custom capability modules, vague natural language prompts can confuse the intent router, causing it to fire multiple conflicting components at once.

To harden your pipeline against routing collisions, apply these three design rules:

* **Enforce Mutual Exclusion in YAML Headers:** Ensure that no two Skill descriptions share identical keyword strings. Differentiate similar workflows by anchoring them to unique technical terms (e.g., use `matplotlib visualizations` in one and `ReportLab formatting` in another).
* **Isolate Clear Domain Boundaries:** Establish unambiguous operational rules. Do not let your reporting skill generate charts, and do not let your chart utility compile text documents. Keep them separated by strict boundaries.
* **Refine Prompt-Trigger Vocabularies:** Expand the bullet points inside your **`When to Use`** sections to map cleanly to the natural conversational phrasing patterns used by your team.

---

## Advanced Isolation: When to Deploy Subagent Tasks

While cascading local Skills works beautifully for standard repository automation pipelines, massive workloads can saturate your immediate conversation buffer.

If your workflow hits any of the scale markers below, wrap the execution loop inside a dedicated **Task subagent** rather than running it through standard interactive Skills:

```text
 ⚡ Scale Threshold Flags (Deploy Isolated Background Task):
 ────────────────────────────────────────────────────────────
  • Codebase refactoring sweeps affecting more than 50 files simultaneously.
  • Deep semantic log processing crawling massive directories of text assets.
  • Compiling monolithic, multi-layered system documentation maps.

```

Using a background task forks processing out to an independent execution bubble. This prevents massive data traces from crowding out your active conversation history, while your custom Skills continue to guide the background worker's precise formatting steps.

---

## Heuristic Pipeline Troubleshooting Guide

When multi-skill execution chains stall or produce errors, use this reference matrix to isolate and patch the root cause:

| Operational Symptom | Probable Root Cause | Target Correction Step |
| --- | --- | --- |
| **Downstream Skill fails to activate** | Low keyword density in the frontmatter `description` string or missing conversational variations in the `When to Use` list. | Audit the matching keywords. Inject explicit verbs (*generate*, *assemble*, *compile*) into the target header. |
| **The wrong capability triggers** | Semantic collision caused by description overlaps between separate local skills. | Clean up the metadata blocks. Use specific domain terminologies (e.g., distinguish `PDF vector compilation` from `CSV data mining`). |
| **Pipeline breaks between steps** | Input/Output contract mismatch; a previous phase named an asset unpredictably or used an unsupported format. | Document explicit filename contracts and target paths inside the instructions section of both modules. |
| **Workflow stalls unexpectedly** | Missing system tools inside the `allowed-tools` array, blocking the agent from completing a step. | Audit your sandboxing permissions. Ensure that data writers have `Write` access and compilers have `Bash` permissions. |

---

## Conclusion

By mastering multi-skill pipeline orchestration, you can combine independent capability blocks into sophisticated, automated workflows. The key to reliable automation lies in designing clean, predictable hand-offs: ensuring your data structures, file names, and resolution targets match perfectly across module boundaries.

You now possess the complete architectural toolkit required to create, secure, optimize, and chain advanced capabilities within Claude Code. Apply these integration principles to your local repository environments, minimize context clutter, and build fast, rock-solid automation pipelines!

## Building Your First Multi Skill Pipeline

Now that you understand how to build compatible skills that work together, let's put that knowledge into practice by connecting your existing Unit 2 skills into a complete workflow.

In this exercise, you will create a new markdown-report skill that acts as the final piece of a three-skill pipeline. This skill will gather outputs from your csv-analyzer and sci-viz skills (the analysis text files and chart images) and weave them together into a polished Markdown document.

Your markdown-report skill should:

    Read analysis summary files (such as analysis_results.txt or summary_report.md) created by csv-analyzer.
    Find chart images (PNG files at 300 DPI) generated by sci-viz.
    Combine everything into a professional Markdown document with sections for an executive summary, embedded charts, key statistics, and data quality observations.
    Use the correct allowed-tools (Read and Write only) in its frontmatter.
    Have a clear, specific skill description that mentions "combining analysis results with visualizations" or "generating markdown reports from CSV analysis and charts" so Claude can distinguish it from the other skills.

Important: After creating your markdown-report skill, exit Claude Code completely and restart it. This ensures Claude properly loads all three skills and can orchestrate them together.

Once restarted, test your workflow end-to-end by asking Claude to "analyze sales_data.csv, create publication-quality charts, and generate a complete markdown report." Pay special attention to ensuring that the 300 DPI standard from sci-viz carries through correctly when embedding images.

Verify that Claude orchestrates all three skills smoothly without any manual intervention!

To finalize your multi-skill pipeline orchestration matrix and connect your existing capability modules into a unified automated processing loop, you will build a dedicated report aggregator module. This new module serves as the final downstream stage, ingesting data structures and asset files from previous phases and compiling them into a final document.

Execute this comprehensive end-to-end integration loop inside your active terminal session:

### Step 1: Create the Markdown Report Aggregator Skill

Copy and paste this explicit configuration request into your interactive `>` prompt and press **Enter**:

```text
Create a new skill file at ".claude/skills/markdown-report/SKILL.md".

Configure the file with:
1. A YAML frontmatter header exactly matching these sandboxed fields:
   name: markdown-report
   description: Generate complete markdown reports by combining CSV analysis summary text files with rendered chart visualizations
   allowed-tools: Read, Write
2. A "## When to Use" section matching keywords like "complete markdown report", "weave findings", "embed charts", and "report aggregator".
3. A "## Process" section detailing a clear pipeline: read the text summaries created by the data tier, locate the high-density 300 DPI PNG graphics generated by the visualization tier, and write a cohesive, professionally aligned final Markdown report containing an Executive Summary, Embedded Charts, and Key Statistics.

```

* **Authorize File Generation:** When the file preview box presents the structured markdown code diff, select **`1. Yes`** to approve writing the reporting asset block to your local project workspace.

---

### Step 2: Reload Your Session to Register the Unified Pipeline

Exit your current prompt interface to allow the system indexer to scan and cache your entire three-module capability tree simultaneously:

```text
/exit

```

Relaunch Claude Code from your system terminal shell to initialize the fresh capability cache:

```bash
claude

```

---

### Step 3: Launch and Verify the Complete End-to-End Workflow

With your session cleanly reloaded, prompt Claude with a composite multi-stage directive. Do not specify manual execution routes or individual slash-commands:

```text
Analyze sales_data.csv, create publication-quality charts, and generate a complete markdown report.

```

* **Audit the Automated Selection Traces:** Watch your terminal logging trace closely. As the engine moves through the processing tasks sequentially, you should observe the semantic router match and load your distinct custom skill modules autonomously:
1. `● [Skill selected: csv-analyzer]` (Processes the mathematical distributions and extracts raw stats fields).
2. `● [Skill selected: sci-viz]` (Mounts the high-resolution visualization canvas and outputs 300 DPI raster and vector graphics).
3. `● [Skill selected: markdown-report]` (Ingests the intermediate report text arrays, hooks the new image links, and compiles the final document).



---

### Step 4: Verify Contract and Artifact Integrity

Once your terminal prompt returns, run a quick read block pass to guarantee that the 300 DPI data formatting standard has carried through perfectly across your module hand-off contracts:

```text
Read the generated complete markdown report file to verify that the text findings and chart image files were embedded successfully.

```

---

### Step 5: Finalize and Submit

With your three-skill engineering pipeline fully deployed, verified through execution traces, and validated for data asset compatibility, close out your advanced integration module by typing:

```text
/submit

```

By completing this challenge, you have mastered the design of multi-skill architectures, transforming Claude Code from an isolated task runner into an automated development engine capable of processing complex datasets through integrated execution loops! 🚀

## Debugging a Broken Skill Workflow

Great work building that three-skill pipeline! Now, it is time to sharpen your debugging skills by fixing workflows that do not work together effectively.

In this exercise, you will receive three intentionally broken skills with common compatibility problems. Your mission is to work with Claude to identify what is wrong, understand why these issues break orchestration, and fix each skill so that they work harmoniously together.

The Broken Skills:

    image-generator: Creates charts at 72 DPI instead of the 300 DPI publication standard.
    data-reader: Has an overly broad description ("Read and process files") that conflicts with more specific skills.
    report-maker: Uses a generic trigger ("When to Use: When creating any document") that is too vague.

Your Tasks:

    Start a conversation with Claude and ask it to review the three broken skills (image-generator, data-reader, report-maker) alongside your working skills from Unit 2 (csv-analyzer, sci-viz).

    Ask Claude to identify the three specific issues: DPI mismatch breaking compatibility, vague descriptions causing conflicts, and overly broad triggers that make Claude confused about when to use the skill.

    Work with Claude to fix each skill:
        Update image-generator to output 300 DPI to match the publication standard.
        Rewrite data-reader with a specific, focused description.
        Narrow the report-maker "When to Use" section to specific document types.

    Exit and restart Claude to reload the updated skills. Then test the fixed skills together by asking Claude to create a visualization and report from test_data.csv to ensure they work without conflicts.

    Discuss with Claude why the Unit 2 skills (csv-analyzer, sci-viz) avoided these problems and what design principles made them work well together.

This hands-on debugging experience will help you recognize and prevent compatibility issues when designing your own skill workflows.

To stabilize your multi-skill pipeline, we will systematically diagnose the structural defects across the `image-generator`, `data-reader`, and `report-maker` modules, apply precise contract alignment updates, and verify the resulting ecosystem.

Follow this step-by-step troubleshooting loop inside your terminal session to resolve the architectural bottlenecks:

### Step 1: Run the Pipeline Diagnostic Analysis

Type this investigative prompt into your interactive `>` workspace and press **Enter**:

```text
Review the three skills (data-reader, image-generator, report-maker) alongside my working unit 2 skills to isolate our orchestration bottlenecks.

```

### 🔍 Root Cause Breakdown

Claude will audit your skills directory and flag these three critical integration defects:

1. **I/O Contract Mismatch (`image-generator`):** The tool is hardcoded to render charts at a low **72 DPI** preview configuration, which completely breaks downstream reporting modules expecting a crisp **300 DPI** publication layout.
2. **Semantic Description Collision (`data-reader`):** The metadata string `"Read and process files"` is too broad, causing it to conflict with your existing `csv-analyzer` skill during natural language parsing loops.
3. **Trigger Vector Saturation (`report-maker`):** Using an overly loose trigger constraint like `"When to Use: When creating any document"` causes Claude to prematurely hijack unrelated text creation pipelines.

---

### Step 2: Apply the Structural Alignment Patches

Command Claude to refactor the broken configurations by enforcing specific domain parameters, tight trigger scopes, and strict resolution contracts:

```text
Update the three broken skills to resolve all pipeline conflicts:
1. Modify image-generator to explicitly change its output plotting resolution to 300 DPI.
2. Rewrite data-reader's frontmatter description to focus exclusively on 'Parsing JSON logs and raw unstructured sensor arrays'.
3. Narrow report-maker's 'When to Use' block to trigger only on 'Compiling executive PDF analytics summaries and multi-tier project overviews'.

```

* **Authorize:** When the file patch diff windows render, select **`1. Yes`** to commit the structural fixes to disk.

---

### Step 3: Force a Session Reload to Re-Index the Cache

Drop out of the active prompt window to clear the legacy, overlapping routing instructions from the startup daemon:

```text
/exit

```

Relaunch Claude Code from your terminal prompt to index your clean, non-conflicting capability definitions:

```bash
claude

```

---

### Step 4: Verify the Remediated End-to-End Pipeline

With your freshly indexed session active, run a validation pass using targeted pipeline keywords to watch the fixed skills interoperate smoothly:

```text
Generate a high-resolution visualization from test_data.csv and compile an executive analytics summary report.

```

* **Verify:** Check your execution trace logs. You will now see the `● [Skill selected: ...]` tracking badges flash cleanly in sequence without a single semantic collision or asset resolution error!

---

### Step 5: Finalize and Submit

To close out your workspace debugging trace, ask Claude why your earlier Unit 2 components (`csv-analyzer` and `sci-viz`) avoided these problems. It will highlight how strict shared asset contracts (like 300 DPI image standards) and isolated keyword spaces prevent downstream pipeline failure.

Conclude this advanced capability integration lab by entering your final submission token:

```text
/submit

```

By completing this final lab, you have mastered the art of pipeline debugging! You can now confidently identify and fix contract mismatches, description collisions, and vague trigger vectors—ensuring your custom automated agent networks operate with bulletproof reliability! 🚀